### Editor for modularizing `avg_vol` function.

In [ ]:
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl
from nilearn import masking # for masking within-brain voxels

import nitools as nt

import os

#_______________________________

# directories
# make as fcn args?
base_dir = '/cifs/diedrichsen/data/smarts_cerebellum'
anat_dir = '/cifs/diedrichsen/data/smarts_cerebellum/anatomicals'

In [ ]:
tissue_dict = {
        'gm': 'c1',
        'wm': 'c2',
        'csf': 'c2'
    }

In [ ]:
def sufficient_weeks(subj_id, subpath, tissue):

    weeks = np.array([0,4,12,24,52])

    p_weeks = []
    for week in weeks:
        #week_path = f'{anat_dir}/{subj_id}/W{week}/wm_results/{subj_id}_W{week}_T1_wm_vol.nii'
        #week_path = week_path
        #week_path = f'{anat_dir}/{subj_id}/W{week}/c2{subj_id}_W{week}_T1.nii' # fix

        # e.g. if in cerebellar_alignment dir
        if not subpath == None:
            base_path =  f'{anat_dir}/{subj_id}/{subpath}/W{week}/'
        else:
            base_path =  f'{anat_dir}/{subj_id}/W{week}/'

        if not tissue==None:
            week_path = f'{base_path}/{tissue_dict[tissue]}{subj_id}_W{week}_T1.nii'
        else:
            week_path = f'{base_path}/{subj_id}_W{week}_T1.nii'


        # skip over missed measurement weeks.
        if not os.path.exists(week_path):
            continue

        p_weeks.append(week)
    
    if len(p_weeks) == 1: # only one measurement week available
        return None # exit function (skip subject)    
    
    return p_weeks

In [ ]:
def sufficient_weeks(subj_id, subpath, tissue = None):

    tissue_dict = {
        'gm': 'c1',
        'wm': 'c2',
        'csf': 'c2'
    }

    weeks = np.array([0,4,12,24,52])

    p_weeks = []
    for week in weeks:
        #week_path = f'{anat_dir}/{subj_id}/W{week}/wm_results/{subj_id}_W{week}_T1_wm_vol.nii'
        #week_path = week_path
        #week_path = f'{anat_dir}/{subj_id}/W{week}/c2{subj_id}_W{week}_T1.nii' # fix

        if not tissue==None:
            week_path = f'{anat_dir}/{subj_id}/{subpath}/W{week}/{tissue_dict[tissue]}{subj_id}_W{week}_T1.nii'
        else:
            week_path = f'{anat_dir}/{subj_id}/{subpath}/W{week}/{subj_id}_W{week}_T1.nii'


        # skip over missed measurement weeks.
        if not os.path.exists(week_path):
            continue

        p_weeks.append(week)
    
    if len(p_weeks) == 1: # only one measurement week available
        return None # exit function (skip subject)
    
    # if sufficient measurement weeks (>1)
    return True

In [ ]:
# this entire loop could probably be its own fucntion.
def response_matrix(Y, 
                    
                    # indices
                    x, y, z, 
                    
                    weeks, subj_id, subpath, tissue):
    
    tissue_dict = {
        'gm': 'c1',
        'wm': 'c2',
        'csf': 'c2'
    }
    

    for week in weeks: # resample ALL weeks, including the reference week
        #week_path = f'{anat_dir}/{subj_id}/W{week}/wm_results/{subj_id}_W{week}_T1_wm_vol.nii'
        #week_path = week_path
        
        #__________________________________________
        # find week image if exists
        if not subpath == None:
            base_path =  f'{anat_dir}/{subj_id}/{subpath}/W{week}/'
        else:
            base_path =  f'{anat_dir}/{subj_id}/W{week}/'

        if not tissue==None:
            week_path = f'{base_path}/{tissue_dict[tissue]}{subj_id}_W{week}_T1.nii'
        else:
            week_path = f'{base_path}/{subj_id}_W{week}_T1.nii'

        # skip over missed measurement weeks.
        if not os.path.exists(week_path):
            continue

        week_img = nib.load(week_path)
        print(f'on week {week} for {subj_id}')
        #__________________________________________

        

        # dummy variable for weeks (code as 0, ..., 4)
        week_dict = {
            '0': 0,
            '4': 1,
            '12': 2,
            '24': 3,
            '52': 4
        }

    
        # resample each week's image so that voxels are exactly on top of reference week voxels; add to response matrix as row vector
        Y[week_dict[str(week)]:,] = nt.sample_image(week_img, # response matrix
                                xm=x, ym = y, zm = z, # world coordinates
                                interpolation = 1 # using trilinear resampling
                                ).flatten() # need to put each week as a row
        
        # now we have Y as a k by p matrix, where k is the number of weeks. 
        # 
    return Y 